# Notebook 2 — Preprocessing & Noise Handling

**Goal:** Implement the full preprocessing pipeline:
1. Noise characterisation (SNR per sensor)
2. Savitzky-Golay smoothing per engine unit
3. Synthetic missing data injection and imputation
4. Per-dataset min-max normalisation (individual scalers)
5. Time-window feature extraction
6. Save processed arrays for use in downstream notebooks


In [ ]:
import sys
sys.path.append('../')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

from src.data_loader    import load_all_datasets, FEATURE_COLS, SENSOR_COLS
from src.preprocessor   import (apply_savgol_filter, inject_missing_data,
                                 impute_missing, fit_normaliser, apply_normaliser,
                                 add_piecewise_rul, find_constant_sensors,
                                 full_preprocess_pipeline, snr_db)
from src.windowing      import create_windows, create_windows_inference

os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models/saved',   exist_ok=True)

datasets = load_all_datasets(data_dir='../data/raw')
print("Datasets loaded:", list(datasets.keys()))


## 2.1 Sensor Noise Characterisation

We measure the Signal-to-Noise Ratio (SNR) in dB for each sensor.
SNR < 20 dB indicates sensors that may benefit from smoothing.


In [ ]:
df_fd001 = datasets['FD001']['train']

snr_scores = {}
for sensor in SENSOR_COLS:
    vals = df_fd001[df_fd001['unit_id'] == 1][sensor].values
    snr_scores[sensor] = snr_db(vals)

snr_series = pd.Series(snr_scores).sort_values()

fig, ax = plt.subplots(figsize=(12, 6))
snr_series.plot(kind='barh', ax=ax, color='coral', edgecolor='black', alpha=0.85)
ax.axvline(20, color='navy', linestyle='--', linewidth=1.5, label='SNR = 20 dB threshold')
ax.set_xlabel('SNR (dB)')
ax.set_title('Signal-to-Noise Ratio per Sensor (FD001, Unit 1)')
ax.legend()
plt.tight_layout()
plt.show()

print("\nSensors below 20 dB (smoothing candidates):")
print(snr_series[snr_series < 20].to_string())


**Insight:** Sensors with SNR below 20 dB contain more noise relative to
their signal. The Savitzky-Golay filter will be applied to all sensors but
is most impactful for low-SNR signals. We use a conservative filter (window=11,
polyorder=3) to smooth while preserving the degradation trend shape.


## 2.2 Savitzky-Golay Filter — Visual Validation


In [ ]:
unit_df   = df_fd001[df_fd001['unit_id'] == 1].copy()
smoothed  = apply_savgol_filter(unit_df, SENSOR_COLS, window_length=11, polyorder=3)

sensor_to_show = 'sensor_11'
residual = unit_df[sensor_to_show].values - smoothed[sensor_to_show].values

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(unit_df['cycle'], unit_df[sensor_to_show],
             alpha=0.6, label='Raw', color='steelblue')
axes[0].plot(smoothed['cycle'], smoothed[sensor_to_show],
             color='red', linewidth=2, label='SG Filtered')
axes[0].set_title(f'{sensor_to_show} — Unit 1 (Raw vs Filtered)')
axes[0].set_xlabel('Cycle')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(unit_df['cycle'], residual, color='gray', alpha=0.7)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].fill_between(unit_df['cycle'], residual, 0, alpha=0.2, color='orange')
axes[1].set_title('Residual (Removed Noise Component)')
axes[1].set_xlabel('Cycle')
axes[1].set_ylabel('Residual Value')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


**Insight:** The residual (noise) oscillates near zero with no systematic
trend, confirming the filter removes only high-frequency noise and leaves
the low-frequency degradation trend intact. The degradation profile's
monotonic shape is fully preserved in the filtered signal.


## 2.3 Synthetic Missing Data — Injection & Imputation


In [ ]:
unit_3_orig    = df_fd001[df_fd001['unit_id'] == 3].copy()
unit_3_missing = inject_missing_data(unit_3_orig, SENSOR_COLS, missing_rate=0.05)
unit_3_imputed = impute_missing(unit_3_missing, SENSOR_COLS, method='linear')

missing_counts = unit_3_missing[SENSOR_COLS].isna().sum()
print("Missing values injected per sensor:")
print(missing_counts[missing_counts > 0].to_string())

for sensor in ['sensor_4', 'sensor_11', 'sensor_14']:
    missing_mask = unit_3_missing[sensor].isna()
    n_missing = missing_mask.sum()
    if n_missing == 0:
        continue

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(unit_3_orig['cycle'], unit_3_orig[sensor],
            linewidth=2, label='Original', alpha=0.8)
    ax.scatter(unit_3_orig.loc[missing_mask, 'cycle'],
               unit_3_orig.loc[missing_mask, sensor],
               color='red', zorder=5, s=50,
               label=f'Injected NaN ({n_missing} pts)')
    ax.plot(unit_3_imputed['cycle'], unit_3_imputed[sensor],
            linestyle='--', color='green', linewidth=1.5, label='Imputed')
    ax.set_title(f'{sensor} — Missing Data Imputation (Unit 3, FD001)')
    ax.set_xlabel('Cycle')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


**Insight:** Linear interpolation accurately recovers missing sensor values
within smooth degradation regions. At series edges, backward fill handles
any remaining NaN values. This dual strategy is robust to both interior
sensor glitches (majority case) and early-lifecycle gaps.


## 2.4 Per-Dataset Normalisation (Independent Scalers)

Each dataset is normalised with its OWN scaler, as specified in Equation 15
of the paper. This deliberately preserves cross-dataset distribution shift,
which is the input to the domain adversarial training process.


In [ ]:
scalers = {}
for ds_id in ['FD001', 'FD002', 'FD003', 'FD004']:
    df_tr  = datasets[ds_id]['train']
    df_te  = datasets[ds_id]['test']

    df_tr_processed, df_te_processed, scaler = full_preprocess_pipeline(
        df_train     = df_tr,
        df_test      = df_te,
        feature_cols = FEATURE_COLS,
        sensor_cols  = SENSOR_COLS,
        smooth       = True,
        max_rul      = 125,
        scaler_save_path = f'../models/saved/scaler_{ds_id}.joblib'
    )

    scalers[ds_id] = scaler
    datasets[ds_id]['train_norm'] = df_tr_processed
    datasets[ds_id]['test_norm']  = df_te_processed

    print(f"{ds_id}: scaler saved, train shape = {df_tr_processed.shape}")


In [ ]:
# Visualise: distributions before vs after normalisation for FD001
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
datasets['FD001']['train'][SENSOR_COLS].hist(
    bins=30, ax=axes[0], layout=(3, 7), figsize=(18, 6))
axes[0].set_title('Before Normalisation')

datasets['FD001']['train_norm'][SENSOR_COLS].hist(
    bins=30, ax=axes[1], layout=(3, 7), figsize=(18, 6))
axes[1].set_title('After Min-Max Normalisation [0, 1]')
plt.tight_layout()
plt.show()


**Insight:** After normalisation, all sensor values are in [0, 1] within
each dataset. Crucially, the SAME raw sensor reading (e.g., sensor_11 = 554)
maps to DIFFERENT normalised values across datasets, preserving the
inter-dataset distribution shift that DANN needs to adapt across.


In [ ]:
# Cross-dataset distribution shift on sensor_11 (post-normalisation)
fig, ax = plt.subplots(figsize=(12, 5))
colors = {'FD001': '#1f77b4', 'FD002': '#ff7f0e',
          'FD003': '#2ca02c', 'FD004': '#d62728'}
for ds_id in ['FD001', 'FD002', 'FD003', 'FD004']:
    vals = datasets[ds_id]['train_norm']['sensor_11']
    vals.hist(bins=60, ax=ax, alpha=0.5, label=ds_id,
              color=colors[ds_id], density=True)
ax.set_title('sensor_11 Normalised Distributions — Cross-Dataset Shift (Preserved)')
ax.set_xlabel('Normalised Value [0, 1]')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()


**Insight:** The normalised distributions of sensor_11 differ noticeably
across datasets. FD002/FD004 show multimodal distributions (6 operating
conditions), while FD001/FD003 show unimodal distributions. This is the
distribution shift the DANN architecture must learn to bridge.


## 2.5 Time-Window Feature Extraction

Implements function h_t from Section 3.2 of the paper.
Each window = (T_w=30) consecutive cycles of sensor readings.


In [ ]:
WINDOW_SIZE = 30

for ds_id in ['FD001', 'FD002', 'FD003', 'FD004']:
    df = datasets[ds_id]['train_norm']
    X, y, unit_ids = create_windows(df, FEATURE_COLS, window_size=WINDOW_SIZE)
    datasets[ds_id]['X_train']   = X
    datasets[ds_id]['y_train']   = y
    datasets[ds_id]['unit_ids']  = unit_ids
    print(f"{ds_id}: X_train={X.shape}, y_train={y.shape}, "
          f"windows per engine ≈ {len(X)//df['unit_id'].nunique():.0f}")


In [ ]:
# Visualise one sample window
sample_window = datasets['FD001']['X_train'][500]   # shape: (30, 24)
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

for i, col in enumerate(FEATURE_COLS[:6]):
    axes[0].plot(sample_window[:, i], label=col, alpha=0.8)
axes[0].set_xlabel('Timestep within Window (0 = oldest)')
axes[0].set_ylabel('Normalised Value')
axes[0].set_title('Sample Time Window — First 6 Features')
axes[0].legend(ncol=3, fontsize=8)
axes[0].grid(alpha=0.3)

# Heatmap of the full window
im = axes[1].imshow(sample_window.T, aspect='auto', cmap='viridis',
                     interpolation='nearest')
axes[1].set_xlabel('Timestep within Window')
axes[1].set_ylabel('Feature Index')
axes[1].set_title('Full Window Heatmap (30 timesteps × 24 features)')
axes[1].set_yticks(range(len(FEATURE_COLS)))
axes[1].set_yticklabels(FEATURE_COLS, fontsize=6)
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.show()


**Insight:** Each window is a 30×24 matrix: 30 cycles of history, 24 features.
The LSTM processes this left-to-right, building a hidden state that captures
how sensor values have evolved over the past 30 cycles before making a RUL
prediction. The heatmap shows how different sensors activate at different
points in the window — key for understanding what temporal patterns the
LSTM learns to detect.


## 2.6 Save Processed Data


In [ ]:
for ds_id in ['FD001', 'FD002', 'FD003', 'FD004']:
    np.save(f'../data/processed/X_train_{ds_id}.npy', datasets[ds_id]['X_train'])
    np.save(f'../data/processed/y_train_{ds_id}.npy', datasets[ds_id]['y_train'])

print("All processed arrays saved to data/processed/")
print("Scalers saved to models/saved/")
